In [1]:
import matplotlib.pyplot as plt
import pandas as pd


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [2]:
dataset = pd.read_csv('busan_accommodation_result.csv') 
dataset.head()
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   장소명     210 non-null    object 
 1   평점      210 non-null    float64
 2   주소      210 non-null    object 
 3   구분      210 non-null    object 
 4   카테고리    210 non-null    object 
 5   연락처     194 non-null    object 
 6   한줄설명    34 non-null     object 
 7   리뷰 개수   210 non-null    object 
 8   태그      210 non-null    object 
 9   가격      210 non-null    object 
 10  영업시간    70 non-null     object 
 11  리뷰      210 non-null    object 
 12  출처      210 non-null    object 
dtypes: float64(1), object(12)
memory usage: 21.5+ KB


In [5]:
dataset["리뷰_길이"] = dataset["리뷰"].astype(str).str.len()


In [6]:
dataset.select_dtypes(include="object") \
  .apply(lambda col: col.astype(str).str.len().max())


장소명        24
주소         44
구분          4
카테고리        6
연락처        14
한줄설명       50
리뷰 개수       5
태그        152
가격        390
영업시간        8
리뷰       1987
출처         10
dtype: int64

In [14]:
# 한국어만 남겨두는 전처리
dataset['한줄설명']

0                                                       NaN
1                                                       NaN
2                                                       NaN
3                                                       NaN
4                                                       NaN
5               관광과 비즈니스의 중심지! 라마다앙코르부산역 호텔에서 편안한 휴식을 즐기세요.
6                                                       NaN
7                                                       NaN
8                                                       NaN
9           매력적인 해운대와 가까운 최적의 위치,\n아늑한 객실과 품격있는 서비스를 이용해보세요
10                                                      NaN
11             탁 트인 바다와 화려한 다이아몬드 브릿지 야경과 함께 배가되는 한화리조트 해운대
12                                                      NaN
13       완벽한 바다 전망과 더불어 편안한 객실을 자랑하는 페어필드 바이 메리어트 송도를 경험하세요
14                                                      NaN
15                                                      NaN
16                                      

In [24]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return text

    # 문자열로 들어간 \n + 실제 공백 문자 모두 제거
    text = re.sub(r'(\\n|\r|\t|\n)', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    # 한글, 영어, 숫자, 기본 문장부호만 유지
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', text)

    return text.strip()


In [28]:
def preprocess_oneline_desc(df, col="한줄설명"):
    df = df.copy()
    df[col] = df[col].apply(clean_text)
    return df

dataset = preprocess_oneline_desc(dataset)


In [ ]:
dataset['한줄설명']
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   장소명     210 non-null    object 
 1   평점      210 non-null    float64
 2   주소      210 non-null    object 
 3   구분      210 non-null    object 
 4   카테고리    210 non-null    object 
 5   연락처     194 non-null    object 
 6   한줄설명    34 non-null     object 
 7   리뷰 개수   210 non-null    object 
 8   태그      210 non-null    object 
 9   가격      210 non-null    object 
 10  영업시간    70 non-null     object 
 11  리뷰      210 non-null    object 
 12  출처      210 non-null    object 
dtypes: float64(1), object(12)
memory usage: 21.5+ KB


In [33]:
dataset['영업시간']
mask = dataset['영업시간'].str.contains('오픈', na=False)
subset = dataset[mask]
subset[['장소명', '영업시간']]


,장소명,영업시간
8,마티에오시리아호텔,08:00 오픈
36,와이컬렉션,09:00 오픈
46,엘시티레지던스 더바인 해운대,09:00 오픈
48,뉴시즈 오시리아 레지던스,09:00 오픈
61,베스트루이스해밀턴호텔 오션테라스,11:00 오픈
78,마리쏠by해운대비치,10:00 오픈
81,뉴시즈 해운대 레지던스,09:00 오픈
91,젬스테이,09:00 오픈
98,오데이즈 오시리아 레지던스,09:00 오픈


In [36]:
edit_df = subset[['영업시간']].copy()


In [37]:
opening_hours1 = [
    {'day': '화', 'hours': '08:00 ~ 22:00'},
    {'day': '수', 'hours': '08:00 ~ 22:00'},
    {'day': '목', 'hours': '08:00 ~ 22:00'},
    {'day': '금', 'hours': '08:00 ~ 22:00'},
    {'day': '토', 'hours': '08:00 ~ 22:00'},
    {'day': '일', 'hours': '08:00 ~ 22:00'},
    {'day': '월', 'hours': '08:00 ~ 22:00'},
    ]

opening_hours2 = [
    {'day': '화', 'hours': '09:00 ~ 02:00'},
    {'day': '수', 'hours': '09:00 ~ 02:00'},
    {'day': '목', 'hours': '09:00 ~ 02:00'},
    {'day': '금', 'hours': '09:00 ~ 02:00'},
    {'day': '토', 'hours': '09:00 ~ 02:00'},
    {'day': '일', 'hours': '09:00 ~ 02:00'},
    {'day': '월', 'hours': '09:00 ~ 02:00'},
    ]

opening_hours3 = [
    {'day': '화', 'hours': '09:00 ~ 22:00'},
    {'day': '수', 'hours': '09:00 ~ 22:00'},
    {'day': '목', 'hours': '09:00 ~ 22:00'},
    {'day': '금', 'hours': '09:00 ~ 22:00'},
    {'day': '토', 'hours': '09:00 ~ 22:00'},
    {'day': '일', 'hours': '09:00 ~ 22:00'},
    {'day': '월', 'hours': '09:00 ~ 22:00'},
    ]


opening_hours4 = [
    {'day': '화', 'hours': '11:00 ~ 20:00'},
    {'day': '수', 'hours': '11:00 ~ 20:00'},
    {'day': '목', 'hours': '11:00 ~ 20:00'},
    {'day': '금', 'hours': '11:00 ~ 20:00'},
    {'day': '토', 'hours': '11:00 ~ 20:00'},
    {'day': '일', 'hours': '11:00 ~ 20:00'},
    {'day': '월', 'hours': '11:00 ~ 20:00'},
    ]

opening_hours5 = [
    {'day': '화', 'hours': '10:00 ~ 22:00'},
    {'day': '수', 'hours': '10:00 ~ 22:00'},
    {'day': '목', 'hours': '10:00 ~ 22:00'},
    {'day': '금', 'hours': '10:00 ~ 22:00'},
    {'day': '토', 'hours': '10:00 ~ 22:00'},
    {'day': '일', 'hours': '10:00 ~ 22:00'},
    {'day': '월', 'hours': '10:00 ~ 22:00'},
    ]


opening_hours6 = [
    {'day': '화', 'hours': '09:00 ~ 23:00'},
    {'day': '수', 'hours': '09:00 ~ 23:00'},
    {'day': '목', 'hours': '09:00 ~ 23:00'},
    {'day': '금', 'hours': '09:00 ~ 23:00'},
    {'day': '토', 'hours': '09:00 ~ 23:00'},
    {'day': '일', 'hours': '09:00 ~ 23:00'},
    {'day': '월', 'hours': '09:00 ~ 23:00'},
    ]


In [38]:
edit_df.loc[edit_df.index[0], '영업시간'] = opening_hours1
edit_df.loc[edit_df.index[1], '영업시간'] = opening_hours2
edit_df.loc[edit_df.index[2], '영업시간'] = opening_hours3 
edit_df.loc[edit_df.index[3], '영업시간'] = opening_hours3
edit_df.loc[edit_df.index[4], '영업시간'] = opening_hours4
edit_df.loc[edit_df.index[5], '영업시간'] = opening_hours5
edit_df.loc[edit_df.index[6], '영업시간'] = opening_hours3
edit_df.loc[edit_df.index[7], '영업시간'] = opening_hours6
edit_df.loc[edit_df.index[8], '영업시간'] = opening_hours3

In [39]:
dataset.loc[edit_df.index, '영업시간'] = edit_df['영업시간']


In [46]:
dataset['영업시간']
mask = dataset['영업시간'].str.contains('까지', na=False)
subset = dataset[mask]
subset[['장소명', '영업시간']]


,장소명,영업시간
41,오션더포인트호텔,12:00 까지
47,더반호텔 해운대점,12:00 까지
66,해운대영무파라드호텔,11:00 까지
75,아크블루호텔,12:00 까지
79,에이치에비뉴 광안해변점,12:00 까지
136,엘시티레지던스 더바인 해운대,22:00 까지
142,더반호텔 해운대점,12:00 까지
147,리자인호텔,12:00 까지
149,젬스테이,23:00 까지
161,영도마리노오토캠핑장,18:00 까지


In [47]:
edit_df = subset[['영업시간']].copy()


In [48]:
opening_hours1 = [
    {'day': '화', 'hours': '16:00 ~ 12:00'},
    {'day': '수', 'hours': '16:00 ~ 12:00'},
    {'day': '목', 'hours': '16:00 ~ 12:00'},
    {'day': '금', 'hours': '16:00 ~ 12:00'},
    {'day': '토', 'hours': '16:00 ~ 12:00'},
    {'day': '일', 'hours': '16:00 ~ 12:00'},
    {'day': '월', 'hours': '16:00 ~ 12:00'},
    ]

opening_hours2 = [
    {'day': '화', 'hours': '15:00 ~ 12:00'},
    {'day': '수', 'hours': '15:00 ~ 12:00'},
    {'day': '목', 'hours': '15:00 ~ 12:00'},
    {'day': '금', 'hours': '15:00 ~ 12:00'},
    {'day': '토', 'hours': '15:00 ~ 12:00'},
    {'day': '일', 'hours': '15:00 ~ 12:00'},
    {'day': '월', 'hours': '15:00 ~ 12:00'},
    ]

opening_hours3 = [
    {'day': '화', 'hours': '16:00 ~ 11:00'},
    {'day': '수', 'hours': '16:00 ~ 11:00'},
    {'day': '목', 'hours': '16:00 ~ 11:00'},
    {'day': '금', 'hours': '16:00 ~ 11:00'},
    {'day': '토', 'hours': '16:00 ~ 11:00'},
    {'day': '일', 'hours': '16:00 ~ 11:00'},
    {'day': '월', 'hours': '16:00 ~ 11:00'},
    ]

opening_hours4 = [
    {'day': '화', 'hours': '09:00 ~ 22:00'},
    {'day': '수', 'hours': '09:00 ~ 22:00'},
    {'day': '목', 'hours': '09:00 ~ 22:00'},
    {'day': '금', 'hours': '09:00 ~ 22:00'},
    {'day': '토', 'hours': '09:00 ~ 22:00'},
    {'day': '일', 'hours': '09:00 ~ 22:00'},
    {'day': '월', 'hours': '09:00 ~ 22:00'},
    ]

opening_hours5 = [
    {'day': '화', 'hours': '18:00 ~ 12:00'},
    {'day': '수', 'hours': '18:00 ~ 12:00'},
    {'day': '목', 'hours': '18:00 ~ 12:00'},
    {'day': '금', 'hours': '18:00 ~ 12:00'},
    {'day': '토', 'hours': '18:00 ~ 12:00'},
    {'day': '일', 'hours': '18:00 ~ 12:00'},
    {'day': '월', 'hours': '18:00 ~ 12:00'},
    ]


opening_hours6 = [
    {'day': '화', 'hours': '09:00 ~ 23:00'},
    {'day': '수', 'hours': '09:00 ~ 23:00'},
    {'day': '목', 'hours': '09:00 ~ 23:00'},
    {'day': '금', 'hours': '09:00 ~ 23:00'},
    {'day': '토', 'hours': '09:00 ~ 23:00'},
    {'day': '일', 'hours': '09:00 ~ 23:00'},
    {'day': '월', 'hours': '09:00 ~ 23:00'},
    ]



opening_hours7 = [
    {'day': '화', 'hours': '09:00 ~ 18:00'},
    {'day': '수', 'hours': '09:00 ~ 18:00'},
    {'day': '목', 'hours': '09:00 ~ 18:00'},
    {'day': '금', 'hours': '09:00 ~ 18:00'},
    {'day': '월', 'hours': '09:00 ~ 18:00'},
    ]

opening_hours8 = [
    {'day': '화', 'hours': '05:00 ~ 20:30'},
    {'day': '수', 'hours': '05:00 ~ 20:30'},
    {'day': '목', 'hours': '05:00 ~ 20:30'},
    {'day': '금', 'hours': '05:00 ~ 20:30'},
    {'day': '토', 'hours': '05:00 ~ 20:30'},
    {'day': '일', 'hours': '05:00 ~ 20:30'},
    {'day': '월', 'hours': '05:00 ~ 20:30'},
    ]


In [49]:
edit_df.loc[edit_df.index[0], '영업시간'] = opening_hours1
edit_df.loc[edit_df.index[1], '영업시간'] = opening_hours2
edit_df.loc[edit_df.index[2], '영업시간'] = opening_hours3
edit_df.loc[edit_df.index[3], '영업시간'] = opening_hours1
edit_df.loc[edit_df.index[4], '영업시간'] = opening_hours1
edit_df.loc[edit_df.index[5], '영업시간'] = opening_hours4
edit_df.loc[edit_df.index[6], '영업시간'] = opening_hours2
edit_df.loc[edit_df.index[7], '영업시간'] = opening_hours5
edit_df.loc[edit_df.index[8], '영업시간'] = opening_hours6
edit_df.loc[edit_df.index[9], '영업시간'] = opening_hours7
edit_df.loc[edit_df.index[10], '영업시간'] = opening_hours8

In [50]:
dataset.loc[edit_df.index, '영업시간'] = edit_df['영업시간']

In [52]:
dataset.head()

,장소명,평점,주소,구분,카테고리,연락처,한줄설명,리뷰 개수,태그,가격,영업시간,리뷰,출처
0,롯데호텔 부산,4.3,부산 부산진구 가야대로 772,숙박업체,호텔,051-810-1000,NaN,116개,"['#2인실', '#가족실', '#물놀이', '#비즈니스호텔', '#사우나', '#실외수영장', '#연회장', '#이동약자접근', '#조식서비스', '#호캉스']","{'룸타입슈페리어 더블 (Superior Double)': '205,700원', '룸타입슈페리어 트윈 (Superior twin)': '205,700원', '룸타입디럭스 더블 (Deluxe Double)': '229,900원', '룸타입디럭스 트윈 (Deluxe Twin)': '229,900원'}",24시간 영업,"['롯데호텔 알바한사람입니다 저도 일을 잘 한 편은 아니지만 , 정말 잘못된 체계라고 생각합니다 . 정직원을 뽑으시던지 계약직을 쓰시던지 알바를 시키면서 계약직을 원하시면 어떻게하자는건지 모르겠습니다 일을 알려주지도 않고 알아서 다 하기를 바라는 신기한 상황을 처음보아 후기를 남깁니다. 알바생을 진짜 직원인것처럼 부려 먹는 신기한 연회장,호텔입니다 ', '연회했는데 직원들이 센스가 하나도 없어서, 쓰레기통 가져다달라, 뜨거운물 가져다달라, 하나하나 말을 해야 겨우 챙겨주고, 서비스는 별1개도 아깝습니다. ', '서면 중심부에 있어서 교통도 편리하고 근처에 맛집도 많아서 좋아요. 호텔은 오래되었지만 관리가 잘되어 있어서 괜찮았어요. 저녁에 실외 수영장도 라이트업 되어서 예뻤고 사진찍기 좋아요. 실내 수영장은 수영하기 좋구요. 라운지 음식도 맛있고 라세느 조식도 맛있어요. 또, 직원분들이 진짜 친절하셔서 좋았어요. ']",카카오맵(2025)
1,시그니엘 부산,4.1,부산 해운대구 달맞이길 30,숙박업체,호텔,051-922-1000,NaN,186개,"['#물놀이', '#사우나', '#야외테라스', '#오션뷰', '#조식서비스', '#호캉스']","{'룸타입그랜드 디럭스 더블 미포하버뷰': '350,900원', '룸타입프리미어 더블 시티뷰': '350,900원', '룸타입프리미어 트윈 미포하버뷰': '399,300원', '룸타입프리미어 패밀리 트윈': '435,600원'}",NaN,"['룸이랑 객실 상태는 완전 좋았고 조식과 주차는 그저 그랬습니다 위치도 좋아서 어디가든지 금방 갈수 있어 좋아요 ', '로비. 엘베 동선과 분위기 어수선. 조식과 인피니티풀 룸 view 좋음. 라운지 협소. 조식 동선이 긴데도불구. 조식과 인피니티풀, 로케이션 때문에 오는듯! 재방문의사 있음. ', '서울 시그니엘 대비 너무 별로임 체크인도 사람 많아서 바글바글... 욕조 물이 안빠지고 이런 일이 자주 있는 듯? 테이블도 까져있고 고급스러운 맛이 없음 ']",카카오맵(2025)
2,신라스테이 해운대,3.6,부산 해운대구 해운대로570번길 46,숙박업체,호텔,051-912-9000,NaN,197개,"['#2인실', '#가족실', '#비즈니스룸', '#비즈니스호텔', '#연회장', '#오션뷰', '#조식서비스', '#호캉스']","{'룸타입[룸온니]스탠다드 더블 시티뷰': '132,165원', '룸타입[룸온니]스탠다드 트윈 시티뷰': '132,165원', '룸타입[룸온니]스탠다드 패밀리 트윈 시티뷰': '132,165원', '룸타입[룸온니]디럭스 더블 시티뷰': '141,955원'}",NaN,"['방에서 욕실을 관람할 수 있도록 벽이 유리벽을 두고 옆으로 열리는 점에서 1차로 놀라고, 전화 쓰는 법이나 화장실 물 내리는 버튼 등등 사소한 것 하나하나 전부 영어로 적어놓고 실내 금연 표시 만큼은 오해없길 바라는 듯 한글로 한번 더 적는게 약간 재섭게 보였는데 침구가 깨끗하고 푹신했던데다 조식 뷔페 직원들이 친절하고 음식이 맛있었어서 5점 남깁니다. ', '업글 해주셔서 주니어 스위트에 들어갔어요 개인적으로 신라스테이는 기대 없이 가는 숙소인데, 주니어 스위트 객실은 좋았어요 ', '해운대 해수욕장 근처 가성비 호텔 해수욕장, 해운대 전통시장 가깝고 좋아요 신라리워즈 포인트 차곡차곡 쌓는중... ']",카카오맵(2025)
3,파크하얏트 부산,4.4,부산 해운대구 마린시티1로 51,숙박업체,호텔,051-990-1234,NaN,178개,"['#사우나', '#실외수영장', '#오션뷰', '#이동약자접근', '#조식서비스', '#호캉스']","{'룸타입스탠다드 (1킹)': '352,000원', '룸타입디럭스 (1킹)': '376,200원', '룸타입디럭스 패밀리 트윈': '376,200원'}",NaN,"['부산 호텔중에 뷰 1등입니다 호텔 리어분들 서비스 너무 좋구요 친절하십니다 조식은 솔직히 아쉬웠습니다 전체적으로나 베이커리류나 시그니엘에 밀리는 편이었습니다만 나쁘지 않았습니다 호텔 규모가 생각보다 작았지만 불편함은 없었습니다 다음에 또 묵을 예정입니다 ', '내 댓글 삭제되었네 왜 지우노 최악 불친절 싸가지 호구되는곳 글 지우면 또 남긴다. ', '다들 너무 친절하시고 객실도 꽤 넓어 좋았습니다 부산에 많은 호텔이 있지만 재방문하고 싶은 호텔 중 하나입니다 위치가 조금 아쉽지만 호캉스 하기엔 충분히 만족스럽네요 ']",카카오맵(2025)
4,그랜드조선 부산,4.3,부산 해운대구 해운대해변로 292,숙박업체,호텔,051-922-5000,NaN,178개,"['#2인실', '#물놀이', '#사우나', '#오션뷰', '#이동약자접근', '#조식서비스', '#호캉스']","{'룸타입슈페리어 시티 더블': '237,600원', '룸타입디럭스 시티 트윈': '231,660원', '룸타입디럭스 시티 킹': '231,660원'}",NaN,"['침구류 좋음 직원들 친절 오션뷰 좋음 복도 방음은 안됨 ', '주차장 들어가는 입구가 너무 협소했지만 아기용품 대여가 많이 돼서 좋았어요.! ', '객실 수에 비해 엘리베이터가 적다. 수영장 크기에 비해 사람이 많다. ']",카카오맵(2025)


In [54]:
# 저장
dataset.to_csv(
    "./final/stay_crawling.csv",
    index=False,
    encoding="utf-8-sig"
)
